In [ ]:
# 프로젝트 경로 설정
from pathlib import Path
import os
import sys

project_root = next(
    (folder for folder in (Path.cwd(), *Path.cwd().parents)
     if (folder / "project_paths.py").is_file()),
    None,
)
if project_root is None:
    raise RuntimeError("프로젝트 폴더에서 노트북을 실행하세요.")
if str(project_root) in sys.path:
    sys.path.remove(str(project_root))
sys.path.insert(0, str(project_root))
sys.modules.pop("project_paths", None)
from project_paths import DATA_ROOT, RAW_DIR, PROCESSED_DIR, OUTPUT_DIR, PATIENT_MAPPING_PATH, deidentified_path

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
import os
import glob
import pandas as pd
from datetime import timedelta

# 경로 설정
table_dir = str(PROCESSED_DIR / 'AEGIS-Administer-Med/de')

# 환자별 노이즈 정보
patient_mapping_df = pd.read_csv(str(PATIENT_MAPPING_PATH), sep=",", quotechar ='"', lineterminator = '\n',dtype = {'pt_id': str})

# 노이즈 매핑 딕셔너리
patient_mapping_df['new_pt_id'] = patient_mapping_df['new_pt_id'].astype(str)

patient_id_map = patient_mapping_df.set_index('pt_id')['new_pt_id'].to_dict()
date_offset_map = patient_mapping_df.set_index('new_pt_id')['date_noise'].astype(int).to_dict()
time_offset_map = patient_mapping_df.set_index('new_pt_id')['time_noise'].astype(int).to_dict()

# 대상 파일 수집
csv_files = glob.glob(os.path.join(table_dir, '**/*.csv'), recursive = True)
print(f"[INFO] 총 처리 대상 파일 수 : {len(csv_files)}")


In [ ]:
# 컬럼 삭제
print("\n[INFO] 컬럼 제거")

columns_to_drop = [
    'act_mt','ward_cd', 'ward_nm'
]

for file in csv_files:
    try:
        medication_administration_df = pd.read_csv(file, dtype = {'pt_id':str}, sep=',', quotechar='"', lineterminator='\n')

        total_cols_to_remove = len(columns_to_drop)
        present_columns = [col for col in columns_to_drop if col in medication_administration_df.columns]
        removed_cols_count = len(present_columns)

        if removed_cols_count > 0:
            medication_administration_df.drop(columns = present_columns, inplace = True)

        medication_administration_df.to_csv(file, index=False, encoding = 'utf-8')

        print(f"[INFO] {file} - 제거 대상: {total_cols_to_remove}개, 실제 제거: {removed_cols_count}개")

    except Exception as e:
        print(f"[ERROR] hos_dt 오류: {file} → {e}")


In [ ]:
# opd_nm, depart, ad_ward에서 신부인과, 정신건강의학과 제거

import os
import glob
import pandas as pd


# 키워드 정의
keywords = "산부인과|정신건강의학과|정신의학과"

# 처리 시작
for file in csv_files:
    try:
        medication_administration_df = pd.read_csv(file, dtype = {'pt_id':str}, sep=',', quotechar='"', lineterminator='\n')
        original_len = len(medication_administration_df)

        # 각 컬럼에 대해 조건에 맞는 행 제거
        condition = pd.Series([False] * len(medication_administration_df))
        for col in ['depart_nm']:
            if col in medication_administration_df.columns:
                condition = condition | medication_administration_df[col].fillna('').str.contains(keywords, regex=True)

        # 해당 행 제거
        medication_administration_df = medication_administration_df[~condition]
        removed = original_len - len(medication_administration_df)

        # 덮어쓰기 저장
        medication_administration_df.to_csv(file, index=False, encoding='utf-8')
        print(f"[INFO] {os.path.basename(file)}: {removed}건 제거 완료")

    except Exception as e:
        print(f"[ERROR] 오류 발생 - {file}: {e}")


In [ ]:
# ① pt_id 치환
print("\n[INFO] ① pt_id 치환 시작")
for file in csv_files:
    try:
        medication_administration_df = pd.read_csv(file, dtype={'pt_id':str},sep=',', quotechar='"', lineterminator='\n')

        if 'pt_id' in medication_administration_df.columns:
            medication_administration_df['pt_id'] = medication_administration_df['pt_id'].map(patient_id_map).fillna(medication_administration_df['pt_id'])

        print(medication_administration_df.head(3))

        medication_administration_df.to_csv(file, index=False, encoding = 'utf-8')
        print(f"[INFO] pt_id 저장 완료: {file}")
    except Exception as e:
        print(f"[ERROR] pt_id 오류: {file} → {e}")


In [ ]:
# ② hos_dt ±7일 ±30분
print("\n[INFO] ② hos_dt 노이즈 적용")
for file in csv_files:
    try:
        medication_administration_df = pd.read_csv(file, dtype = {'pt_id':str}, sep=',', quotechar='"', lineterminator='\n')

        if 'hos_dt' in medication_administration_df.columns:
            medication_administration_df['hos_dt'] = pd.to_datetime(medication_administration_df['hos_dt'], errors='coerce')
            medication_administration_df['hos_dt'] = medication_administration_df['hos_dt'] + medication_administration_df['pt_id'].map(lambda x: timedelta(days=date_offset_map.get(x, 0), minutes=time_offset_map.get(x, 0)))

            print(medication_administration_df.head(3))

            medication_administration_df.to_csv(file, index=False, encoding = 'utf-8')
            print(f"[INFO] hos_dt 저장 완료: {file}")
    except Exception as e:
        print(f"[ERROR] hos_dt 오류: {file} → {e}")


In [ ]:
# order_date 처리

for file in csv_files:

    try:

        medication_administration_df = pd.read_csv(file, dtype={'pt_id': str}, sep=",", quotechar ='"', lineterminator = '\n')


        if 'order_date' in medication_administration_df.columns:

            medication_administration_df['order_date'] = pd.to_datetime(medication_administration_df['order_date'], errors = 'coerce')



            medication_administration_df['order_date'] = medication_administration_df['order_date'] \
                + pd.to_timedelta(medication_administration_df['pt_id'].map(date_offset_map).fillna(0).astype(int), unit = "D")


            print(medication_administration_df.head(3))



            medication_administration_df.to_csv(file, index = False, encoding = 'utf-8')

            print(f"[INFO] order_date 완료: {file}")





    except Exception as e:

        print(f"[ERROR] order_date 오류: {file} -> {e}")


In [ ]:
#  ad_dt ±7일 ±30분
print("\n[INFO] ad_dt 노이즈 적용")
for file in csv_files:
    try:
        medication_administration_df = pd.read_csv(file, dtype = {'pt_id':str}, sep=',', quotechar='"', lineterminator='\n')

        if 'ad_dt' in medication_administration_df.columns:
            medication_administration_df['ad_dt'] = pd.to_datetime(medication_administration_df['ad_dt'], errors='coerce')
            medication_administration_df['ad_dt'] = medication_administration_df['ad_dt'] + medication_administration_df['pt_id'].map(lambda x: timedelta(days=date_offset_map.get(x, 0), minutes=time_offset_map.get(x, 0)))

            print(medication_administration_df.head(3))

            medication_administration_df.to_csv(file, index=False, encoding = 'utf-8')
            print(f"[INFO] ad_dt 저장 완료: {file}")
    except Exception as e:
        print(f"[ERROR] ad_dt 오류: {file} → {e}")


In [ ]:
# act_reserv_date 처리

for file in csv_files:

    try:

        medication_administration_df = pd.read_csv(file, dtype={'pt_id': str}, sep=",", quotechar ='"', lineterminator = '\n')


        if 'act_reserv_date' in medication_administration_df.columns:

            medication_administration_df['act_reserv_date'] = pd.to_datetime(medication_administration_df['act_reserv_date'], errors = 'coerce')



            medication_administration_df['act_reserv_date'] = medication_administration_df['act_reserv_date'] \
                + pd.to_timedelta(medication_administration_df['pt_id'].map(date_offset_map).fillna(0).astype(int), unit = "D")


            print(medication_administration_df.head(3))



            medication_administration_df.to_csv(file, index = False, encoding = 'utf-8')

            print(f"[INFO] act_reserv_date 완료: {file}")





    except Exception as e:

        print(f"[ERROR] act_reserv_date 오류: {file} -> {e}")


In [ ]:
#  act_schedue_dt ±7일 ±30분
print("\n[INFO] act_schedue_dt 노이즈 적용")
for file in csv_files:
    try:
        medication_administration_df = pd.read_csv(file, dtype = {'pt_id':str}, sep=',', quotechar='"', lineterminator='\n')

        if 'act_schedue_dt' in medication_administration_df.columns:
            medication_administration_df['act_schedue_dt'] = pd.to_datetime(medication_administration_df['act_schedue_dt'], errors='coerce')
            medication_administration_df['act_schedue_dt'] = medication_administration_df['act_schedue_dt'] + medication_administration_df['pt_id'].map(lambda x: timedelta(days=date_offset_map.get(x, 0), minutes=time_offset_map.get(x, 0)))

            print(medication_administration_df.head(3))

            medication_administration_df.to_csv(file, index=False, encoding = 'utf-8')
            print(f"[INFO] act_schedue_dt 저장 완료: {file}")
    except Exception as e:
        print(f"[ERROR] act_schedue_dt 오류: {file} → {e}")


In [ ]:
#  act_dt ±7일 ±30분
print("\n[INFO] act_dt 노이즈 적용")
for file in csv_files:
    try:
        medication_administration_df = pd.read_csv(file, dtype = {'pt_id':str}, sep=',', quotechar='"', lineterminator='\n')

        if 'act_dt' in medication_administration_df.columns:
            medication_administration_df['act_dt'] = pd.to_datetime(medication_administration_df['act_dt'], errors='coerce')
            medication_administration_df['act_dt'] = medication_administration_df['act_dt'] + medication_administration_df['pt_id'].map(lambda x: timedelta(days=date_offset_map.get(x, 0), minutes=time_offset_map.get(x, 0)))

            print(medication_administration_df.head(3))

            medication_administration_df.to_csv(file, index=False, encoding = 'utf-8')
            print(f"[INFO] act_dt 저장 완료: {file}")
    except Exception as e:
        print(f"[ERROR] act_dt 오류: {file} → {e}")


In [ ]:
#  act_edit_dt ±7일 ±30분
print("\n[INFO] act_edit_dt 노이즈 적용")
for file in csv_files:
    try:
        medication_administration_df = pd.read_csv(file, dtype = {'pt_id':str}, sep=',', quotechar='"', lineterminator='\n')

        if 'act_edit_dt' in medication_administration_df.columns:
            medication_administration_df['act_edit_dt'] = pd.to_datetime(medication_administration_df['act_edit_dt'], errors='coerce')
            medication_administration_df['act_edit_dt'] = medication_administration_df['act_edit_dt'] + medication_administration_df['pt_id'].map(lambda x: timedelta(days=date_offset_map.get(x, 0), minutes=time_offset_map.get(x, 0)))

            print(medication_administration_df.head(3))

            medication_administration_df.to_csv(file, index=False, encoding = 'utf-8')
            print(f"[INFO] act_edit_dt 저장 완료: {file}")
    except Exception as e:
        print(f"[ERROR] act_edit_dt 오류: {file} → {e}")


In [ ]:
#  act_nm 기본처방 데이터 제거
print("\n[INFO] act_nm 기본처방 데이터 제거")
for file in csv_files:
    try:
        medication_administration_df = pd.read_csv(file, dtype = {'pt_id':str}, sep=',', quotechar='"', lineterminator='\n')

        if 'act_nm' in medication_administration_df.columns:
            medication_administration_df = medication_administration_df[medication_administration_df['act_nm'] != '기본처방']

            print(f"[INFO] 처리 중: {os.path.basename(file)}")

            medication_administration_df.to_csv(file, index=False, encoding = 'utf-8-sig')
            print(f"[INFO] act_edit_dt 저장 완료: {file}")
    except Exception as e:
        print(f"[ERROR] act_edit_dt 오류: {file} → {e}")
